# Predicción de pasajeros en vuelos regulares de cabotaje - ANAC 2025

Esta notebook contiene la descripción y preparación del dataset que se utilizará en el proyecto de Aprendizaje Automático.

## 1. Carga del dataset

Se carga el archivo correspondiente a los movimientos aéreos registrados por ANAC durante el año 2025 y se muestran las primeras filas para tener una primera vista de los datos.

In [23]:
import pandas as pd
df = pd.read_csv(
    "../data/raw/informe-ministerio-actualizado-dic-final.csv",
    sep=";",
    decimal=","
)

df.head()

,Fecha UTC,Hora UTC,Clase de Vuelo (todos los vuelos),Clasificación Vuelo,Tipo de Movimiento,Aeropuerto,Origen / Destino,Aerolinea Nombre,Aeronave,Pasajeros,PAX,Calidad dato
0,01/01/2025,0:00,Regular,Doméstico,Aterrizaje,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,167.0,83.5,DEFINITIVO
1,01/01/2025,0:03,Regular,Internacional,Despegue,EZE,KATL,DELTA AIRLINES,0,281.0,281.0,DEFINITIVO
2,01/01/2025,0:06,Regular,Doméstico,Aterrizaje,SAL,NEU,JETSMART AIRLINES S.A.,AIB-A320-232,174.0,87.0,DEFINITIVO
3,01/01/2025,0:07,Regular,Doméstico,Aterrizaje,CBA,BAR,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8U3,83.0,41.5,DEFINITIVO
4,01/01/2025,0:07,Regular,Internacional,Despegue,AER,SCEL,LATAM AIRLINES GROUP S.A.(exLAN AIRLINE),0,162.0,162.0,DEFINITIVO


## 2. Descripción del dataset

### 2.1 Cantidad de instancias y columnas

Se revisa la cantidad total de registros y columnas que contiene el dataset para conocer su tamaño.

In [24]:
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

Cantidad de filas: 597784
Cantidad de columnas: 12


### 2.2 Columnas y tipos de datos

Se revisan las columnas disponibles en el dataset y el tipo de dato asociado a cada una. Esto permite identificar qué información contiene cada registro y detectar si alguna variable necesita ser transformada antes de utilizarla en el proyecto.

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 597784 entries, 0 to 597783
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Fecha UTC                          597784 non-null  str    
 1   Hora UTC                           597784 non-null  str    
 2   Clase de Vuelo (todos los vuelos)  597784 non-null  str    
 3   Clasificación Vuelo                597784 non-null  str    
 4   Tipo de Movimiento                 597784 non-null  str    
 5   Aeropuerto                         597784 non-null  str    
 6   Origen / Destino                   597784 non-null  str    
 7   Aerolinea Nombre                   597784 non-null  str    
 8   Aeronave                           597784 non-null  str    
 9   Pasajeros                          597784 non-null  float64
 10  PAX                                597784 non-null  float64
 11  Calidad dato                       597784 non-null

### 2.3 Descripción de las variables

A partir de la documentación oficial del dataset publicada por la Administración Nacional de Aviación Civil (ANAC), se describe la información contenida en cada una de las variables.

| Variable | Descripción |
|---|---|
| `Fecha UTC` | Fecha en la que se registró la operación aérea, expresada en horario UTC. |
| `Hora UTC` | Hora de la operación expresada en Tiempo Universal Coordinado (UTC). |
| `Clase de Vuelo (todos los vuelos)` | Categoría a la que pertenece el vuelo, como Regular, No Regular, Vuelo Escuela, entre otras. |
| `Clasificación Vuelo` | Clasificación del vuelo según corresponda a una operación de cabotaje o internacional. |
| `Tipo de Movimiento` | Indica si el movimiento registrado corresponde a un aterrizaje o a un despegue. |
| `Aeropuerto` | Código FAA del aeropuerto en el que se registra la operación. |
| `Origen / Destino` | Código del aeropuerto relacionado con el origen o destino de la operación. Para aeropuertos nacionales se utiliza el código FAA y para aeropuertos extranjeros el código OACI. |
| `Aerolinea Nombre` | Nombre de la línea aérea que realiza la operación. |
| `Aeronave` | Código utilizado para identificar al fabricante y al modelo específico de la aeronave. |
| `Pasajeros` | Cantidad de pasajeros correspondiente a una operación individual de despegue o aterrizaje. En vuelos de cabotaje se registran ambos movimientos. |
| `PAX` | Cantidad asociada a tarjetas de embarque, donde 1 PAX equivale a una tarjeta de embarque. La documentación indica que esta variable se utiliza para analizar pasajeros de cabotaje a nivel total del país. |
| `Calidad dato` | Indica la calidad o condición del dato publicado. La fuente señala que los registros más recientes pueden tener carácter provisorio. |

## 3. Origen del dataset

Los datos utilizados en este proyecto provienen del portal de Datos Abiertos de la Secretaría de Transporte de la Nación y fueron publicados por la Administración Nacional de Aviación Civil (ANAC).

El recurso utilizado se denomina **“Aterrizajes y despegues procesados por la Administración Nacional de Aviación Civil”** y contiene información sobre movimientos aéreos registrados en aeropuertos de Argentina.

Para este proyecto se utiliza el archivo correspondiente al año **2025**, descargado en formato CSV desde la fuente oficial.

**Fuente:** Administración Nacional de Aviación Civil (ANAC) - Datos Abiertos de Transporte.

**Dataset:** Aterrizajes y despegues procesados por la Administración Nacional de Aviación Civil.

**Período utilizado:** año 2025.

## 4. Definición de los datos para el proyecto

### 4.1 Variable objetivo

La variable objetivo del proyecto será `Pasajeros`, ya que representa la cantidad de pasajeros correspondiente a una operación aérea.

El objetivo del modelo será estimar este valor para vuelos regulares de cabotaje, utilizando únicamente información que pueda estar disponible antes de realizar la predicción.

La columna `PAX` no se utilizará como variable predictora hasta confirmar su significado y verificar que no contenga información derivada directa o indirectamente de la cantidad de pasajeros.

### 4.2 Momento de la predicción

La idea es realizar la predicción antes de que salga el vuelo. De esta forma, la estimación de pasajeros podría servir para anticipar cuántas personas se espera transportar en una operación determinada.

Por eso, solo deberían utilizarse como variables predictoras aquellas que estén disponibles antes de que el vuelo se realice.

En el caso de `Fecha UTC` y `Hora UTC`, primero hay que revisar qué representan exactamente dentro del dataset, ya que es necesario confirmar si corresponden a información conocida de antemano o al momento real en que ocurrió la operación. Esto es importante para evitar usar datos que no estarían disponibles al momento de hacer la predicción.

### 4.3 Qué representa una observación

Cada fila del dataset representa un movimiento aéreo registrado por ANAC. Ese movimiento puede ser un despegue o un aterrizaje, según el valor de la variable `Tipo de Movimiento`.

Esto es importante porque un mismo vuelo podría aparecer más de una vez en la base: por ejemplo, como despegue en un aeropuerto y como aterrizaje en otro.

Antes de preparar el dataset definitivo para el modelo, se va a revisar cómo están registrados estos movimientos y cómo se relacionan las columnas `Aeropuerto` y `Origen / Destino` con cada tipo de movimiento.

A partir de esa revisión se definirá una única forma de representar cada operación, para evitar contar dos veces el mismo vuelo o generar registros duplicados entre los datos de entrenamiento y evaluación.

### 4.4 Variables disponibles para la predicción

Como la predicción se plantea antes de la salida del vuelo, las variables predictoras tienen que corresponder a información que pueda conocerse en ese momento.

Como punto de partida, se consideran las siguientes variables:

- `Aeropuerto` y `Origen / Destino`, una vez que se defina correctamente cuál representa el origen y cuál el destino según el tipo de movimiento.
- `Aerolinea Nombre`, ya que identifica a la aerolínea encargada de la operación.
- `Fecha UTC` y `Hora UTC`, siempre que se confirme que esa información puede conocerse antes de la salida del vuelo.
- `Aeronave`, si se comprueba que el tipo de aeronave está definido y disponible antes de realizar la predicción.

Por otro lado, `Pasajeros` será la variable objetivo y no se utilizará como entrada del modelo. La variable `PAX` también debe revisarse antes de decidir su uso, ya que podría contener información relacionada directamente con la cantidad de pasajeros y generar fuga de información.

Las variables `Clase de Vuelo (todos los vuelos)`, `Clasificación Vuelo` y `Tipo de Movimiento` se utilizarán principalmente para seleccionar las operaciones que correspondan al alcance del proyecto. `Calidad dato` se considera información asociada al registro y se revisará antes de decidir si aporta información útil para la predicción.

## 5. Exploración y preparación de los datos

### 5.1 Valores faltantes

Antes de empezar a filtrar o transformar el dataset, se revisa si existen valores faltantes en alguna de las columnas.

Esto permite detectar si hay datos incompletos que necesiten algún tratamiento antes de continuar con la preparación del conjunto de datos.


In [26]:
df.isna().sum()

Fecha UTC                            0
Hora UTC                             0
Clase de Vuelo (todos los vuelos)    0
Clasificación Vuelo                  0
Tipo de Movimiento                   0
Aeropuerto                           0
Origen / Destino                     0
Aerolinea Nombre                     0
Aeronave                             0
Pasajeros                            0
PAX                                  0
Calidad dato                         0
dtype: int64

#### Resultado

No se encontraron valores faltantes en ninguna de las columnas del dataset. Por este motivo, no es necesario aplicar ningún tratamiento relacionado con datos nulos antes de continuar con la preparación.

### 5.2 Registros duplicados

También se revisa si existen filas duplicadas dentro del dataset.

Antes de eliminar cualquier registro, es importante comprobar si realmente se trata de un dato repetido o si puede corresponder a otra operación con los mismos valores.

In [27]:
df.duplicated().sum()

np.int64(814)

#### Revisión de los registros duplicados

Se encontraron 814 filas duplicadas en el dataset original. Antes de decidir si corresponde eliminarlas, se revisan algunos de estos registros para comprobar si realmente representan datos repetidos o si podrían corresponder a operaciones distintas.

In [28]:
df[df.duplicated(keep=False)].head(20)

,Fecha UTC,Hora UTC,Clase de Vuelo (todos los vuelos),Clasificación Vuelo,Tipo de Movimiento,Aeropuerto,Origen / Destino,Aerolinea Nombre,Aeronave,Pasajeros,PAX,Calidad dato
7759,06/01/2025,11:59,Vuelo Oficial Nacional,Doméstico,Despegue,RTA,TRC,0,0,0.0,0.0,DEFINITIVO
7760,06/01/2025,11:59,Vuelo Oficial Nacional,Doméstico,Despegue,RTA,TRC,0,0,0.0,0.0,DEFINITIVO
8234,06/01/2025,16:43,Vuelo Oficial Nacional,Doméstico,Aterrizaje,RTA,TRC,0,0,0.0,0.0,DEFINITIVO
8235,06/01/2025,16:43,Vuelo Oficial Nacional,Doméstico,Aterrizaje,RTA,TRC,0,0,0.0,0.0,DEFINITIVO
12402,09/01/2025,11:24,Vuelo Oficial Nacional,Doméstico,Despegue,RTA,RTA,0,0,0.0,0.0,DEFINITIVO
12403,09/01/2025,11:24,Vuelo Oficial Nacional,Doméstico,Despegue,RTA,RTA,0,0,0.0,0.0,DEFINITIVO
12565,09/01/2025,12:58,Vuelo Oficial Nacional,Doméstico,Aterrizaje,RTA,RTA,0,0,0.0,0.0,DEFINITIVO
12566,09/01/2025,12:58,Vuelo Oficial Nacional,Doméstico,Aterrizaje,RTA,RTA,0,0,0.0,0.0,DEFINITIVO
15016,10/01/2025,20:56,Vuelo Escuela,Doméstico,Aterrizaje,MOR,MOR,0,CE-152,0.0,0.0,DEFINITIVO
15017,10/01/2025,20:56,Vuelo Escuela,Doméstico,Aterrizaje,MOR,MOR,0,CE-152,0.0,0.0,DEFINITIVO


#### Resultado

En el dataset original se encontraron 814 filas duplicadas. Al revisar algunos casos, se observa que varios corresponden a clases de vuelo que no forman parte del alcance del proyecto, como vuelos oficiales o vuelos escuela.

Por este motivo, en esta etapa no se eliminan registros duplicados. Primero se seleccionarán los vuelos regulares de cabotaje y luego se volverá a revisar la existencia de duplicados dentro del conjunto de datos que realmente se utilizará en el proyecto.

### 5.3 Selección de vuelos regulares de cabotaje

Como el proyecto está enfocado únicamente en vuelos regulares de cabotaje, se revisan las categorías disponibles en `Clase de Vuelo (todos los vuelos)` y `Clasificación Vuelo` antes de aplicar el filtro.

In [29]:
df["Clase de Vuelo (todos los vuelos)"].value_counts()

Clase de Vuelo (todos los vuelos)
Regular                                   364274
Vuelo Escuela                              78885
Vuelo Privado con Matrícula Nacional       71533
No Regular                                 47875
Vuelo Oficial Nacional                     25005
Vuelo Privado con Matrícula Extranjera      4948
Vuelo de Adiestramiento                     3211
Trabajo Aéreo                               1424
Vuelo Oficial Extranjero                     629
Name: count, dtype: int64

In [30]:
df["Clasificación Vuelo"].value_counts()

Clasificación Vuelo
Doméstico        470037
Internacional    127747
Name: count, dtype: int64

#### Filtrado de los registros

A partir de las categorías observadas, se seleccionan únicamente los registros cuya clase de vuelo es `Regular` y cuya clasificación es `Doméstico`, ya que son los que corresponden al alcance definido para el proyecto.

El resto de las clases de vuelo y los vuelos internacionales no se van a utilizar para el análisis ni para el posterior entrenamiento del modelo.

In [31]:
df_vuelos = df[
    (df["Clase de Vuelo (todos los vuelos)"] == "Regular") &
    (df["Clasificación Vuelo"] == "Doméstico")
].copy()

print("Cantidad de filas:", df_vuelos.shape[0])
print("Cantidad de columnas:", df_vuelos.shape[1])

Cantidad de filas: 265624
Cantidad de columnas: 12


### 5.4 Revisión del tipo de movimiento

Después de seleccionar los vuelos regulares de cabotaje, se revisa cómo están distribuidos los registros según el tipo de movimiento.

Esto es necesario porque un mismo vuelo podría aparecer registrado como despegue en un aeropuerto y como aterrizaje en otro. Antes de decidir qué registros se van a utilizar, primero hay que entender cómo está representada esta información en el dataset.

In [32]:
df_vuelos["Tipo de Movimiento"].value_counts()

Tipo de Movimiento
Despegue      132846
Aterrizaje    132778
Name: count, dtype: int64

#### Interpretación de `Aeropuerto` y `Origen / Destino`

Para entender cómo se representa cada operación, se revisan algunos registros de despegues y aterrizajes por separado.

El objetivo es comprobar cómo deben interpretarse las columnas `Aeropuerto` y `Origen / Destino` según el tipo de movimiento antes de decidir qué registros se van a utilizar en el dataset final.

In [33]:
df_vuelos[
    df_vuelos["Tipo de Movimiento"] == "Despegue"
][["Fecha UTC", "Hora UTC", "Aeropuerto", "Origen / Destino", "Aerolinea Nombre", "Pasajeros"]].head(10)

,Fecha UTC,Hora UTC,Aeropuerto,Origen / Destino,Aerolinea Nombre,Pasajeros
6,01/01/2025,0:09,IGU,EZE,AEROLINEAS ARGENTINAS SA,83.0
8,01/01/2025,0:12,DOZ,AER,JETSMART AIRLINES S.A.,23.0
12,01/01/2025,0:20,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,103.0
15,01/01/2025,0:29,IGU,EZE,FB LÍNEAS AÉREAS - FLYBONDI,104.0
21,01/01/2025,0:44,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,103.0
24,01/01/2025,0:46,NEU,CBA,FB LÍNEAS AÉREAS - FLYBONDI,76.0
25,01/01/2025,0:46,SAL,EZE,JETSMART AIRLINES S.A.,77.0
27,01/01/2025,0:53,CBA,AER,FB LÍNEAS AÉREAS - FLYBONDI,57.0
35,01/01/2025,1:14,NEU,EZE,JETSMART AIRLINES S.A.,41.0
36,01/01/2025,1:15,USU,EZE,AEROLINEAS ARGENTINAS SA,117.0


In [34]:
df_vuelos[
    df_vuelos["Tipo de Movimiento"] == "Aterrizaje"
][["Fecha UTC", "Hora UTC", "Aeropuerto", "Origen / Destino", "Aerolinea Nombre", "Pasajeros"]].head(10)

,Fecha UTC,Hora UTC,Aeropuerto,Origen / Destino,Aerolinea Nombre,Pasajeros
0,01/01/2025,0:00,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,167.0
2,01/01/2025,0:06,SAL,NEU,JETSMART AIRLINES S.A.,174.0
3,01/01/2025,0:07,CBA,BAR,FB LÍNEAS AÉREAS - FLYBONDI,83.0
5,01/01/2025,0:08,EZE,USU,AEROLINEAS ARGENTINAS SA,123.0
7,01/01/2025,0:11,EZE,ECA,AEROLINEAS ARGENTINAS SA,154.0
10,01/01/2025,0:16,USU,AER,AEROLINEAS ARGENTINAS SA,134.0
11,01/01/2025,0:20,EZE,JUJ,FB LÍNEAS AÉREAS - FLYBONDI,174.0
13,01/01/2025,0:25,AER,POS,FB LÍNEAS AÉREAS - FLYBONDI,170.0
14,01/01/2025,0:25,EZE,USU,FB LÍNEAS AÉREAS - FLYBONDI,156.0
22,01/01/2025,0:45,AER,DOZ,FB LÍNEAS AÉREAS - FLYBONDI,81.0


#### Resultado de la revisión

A partir de la documentación oficial y de los registros que se observaron, se puede ver que el significado de `Aeropuerto` y `Origen / Destino` cambia según el tipo de movimiento.

En los despegues, `Aeropuerto` representa el aeropuerto desde donde sale el vuelo y `Origen / Destino` indica hacia dónde se dirige. En los aterrizajes pasa lo contrario: `Aeropuerto` corresponde al aeropuerto de llegada y `Origen / Destino` indica de dónde viene el vuelo.

Como ANAC registra tanto el despegue como el aterrizaje en los vuelos de cabotaje, mantener ambos tipos de movimiento podría hacer que un mismo vuelo aparezca representado dos veces.

Para trabajar de una forma más consistente, en este proyecto se van a usar solamente los registros correspondientes a despegues. De esta manera también se simplifica la interpretación de las columnas de origen y destino.

In [35]:
df_vuelos = df_vuelos[
    df_vuelos["Tipo de Movimiento"] == "Despegue"
].copy()

print("Cantidad de filas:", df_vuelos.shape[0])
print("Cantidad de columnas:", df_vuelos.shape[1])

Cantidad de filas: 132846
Cantidad de columnas: 12


### 5.5 Revisión de duplicados después del filtrado

Una vez seleccionados los vuelos regulares de cabotaje y solamente los despegues, se vuelve a revisar si quedan registros duplicados dentro del conjunto de datos que se va a utilizar.

La idea es analizar los duplicados sobre este subconjunto y no sobre el dataset completo, ya que muchas de las filas repetidas que aparecían al principio pertenecían a otros tipos de vuelo.

In [36]:
df_vuelos.duplicated().sum()

np.int64(8)

#### Revisión de los duplicados encontrados

Después de aplicar los filtros todavía aparecen 8 registros duplicados. Como ahora se está trabajando solamente con las operaciones que forman parte del proyecto, se revisan estos casos antes de decidir si corresponde eliminarlos.

In [37]:
df_vuelos[df_vuelos.duplicated(keep=False)]

,Fecha UTC,Hora UTC,Clase de Vuelo (todos los vuelos),Clasificación Vuelo,Tipo de Movimiento,Aeropuerto,Origen / Destino,Aerolinea Nombre,Aeronave,Pasajeros,PAX,Calidad dato
46556,30/01/2025,13:41,Regular,Doméstico,Despegue,TUC,CBA,AEROLINEAS ARGENTINAS SA,EMB-ERJ190100IGW,78.0,39.0,DEFINITIVO
46557,30/01/2025,13:41,Regular,Doméstico,Despegue,TUC,CBA,AEROLINEAS ARGENTINAS SA,EMB-ERJ190100IGW,78.0,39.0,DEFINITIVO
48278,31/01/2025,12:59,Regular,Doméstico,Despegue,MDP,AER,AEROLINEAS ARGENTINAS SA,BO-737-800,170.0,85.0,DEFINITIVO
48279,31/01/2025,12:59,Regular,Doméstico,Despegue,MDP,AER,AEROLINEAS ARGENTINAS SA,BO-737-800,170.0,85.0,DEFINITIVO
320875,20/07/2025,0:32,Regular,Doméstico,Despegue,EZE,BAR,ANDES LÍNEAS AÉREAS S.A.,BO-737-7K2,82.0,41.0,DEFINITIVO
320876,20/07/2025,0:32,Regular,Doméstico,Despegue,EZE,BAR,ANDES LÍNEAS AÉREAS S.A.,BO-737-7K2,82.0,41.0,DEFINITIVO
359656,11/08/2025,20:42,Regular,Doméstico,Despegue,AER,SAL,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,159.0,79.5,DEFINITIVO
359657,11/08/2025,20:42,Regular,Doméstico,Despegue,AER,SAL,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,159.0,79.5,DEFINITIVO
370690,17/08/2025,23:18,Regular,Doméstico,Despegue,AER,UIS,AEROLINEAS ARGENTINAS SA,EMB-ERJ190100IGW,92.0,46.0,DEFINITIVO
370691,17/08/2025,23:18,Regular,Doméstico,Despegue,AER,UIS,AEROLINEAS ARGENTINAS SA,EMB-ERJ190100IGW,92.0,46.0,DEFINITIVO


#### Tratamiento de los registros duplicados

Después de revisar los casos encontrados, se observa que los registros aparecen repetidos con los mismos valores en todas las columnas disponibles.

Como no existe un identificador único de vuelo en el dataset, no es posible comprobar de forma independiente cada caso. Sin embargo, al tratarse de filas completamente iguales dentro del conjunto ya filtrado, se consideran registros duplicados y se elimina una de las copias para evitar que una misma observación tenga más peso en el análisis y en el futuro entrenamiento del modelo.

In [38]:
df_vuelos = df_vuelos.drop_duplicates().copy()

print("Cantidad de filas después de eliminar duplicados:", df_vuelos.shape[0])
print("Duplicados restantes:", df_vuelos.duplicated().sum())

Cantidad de filas después de eliminar duplicados: 132838
Duplicados restantes: 0


### 5.6 Revisión de la variable PAX

Antes de decidir si `PAX` puede utilizarse como variable predictora, se revisa su relación con `Pasajeros`.

Esto es importante porque `Pasajeros` es la variable que se quiere predecir. Si `PAX` contiene la misma información o un valor derivado directamente de ella, utilizarla como entrada del modelo produciría fuga de información.

In [39]:
df_vuelos[["Pasajeros", "PAX"]].head(20)

,Pasajeros,PAX
6,83.0,41.5
8,23.0,11.5
12,103.0,51.5
15,104.0,52.0
21,103.0,51.5
24,76.0,38.0
25,77.0,38.5
27,57.0,28.5
35,41.0,20.5
36,117.0,58.5


In [40]:
(df_vuelos["PAX"] == df_vuelos["Pasajeros"] / 2).value_counts()

True    132838
Name: count, dtype: int64

#### Resultado de la revisión

Al revisar todos los registros del conjunto filtrado, se comprobó que `PAX` es exactamente la mitad de `Pasajeros` en los 132.838 casos.

Como `Pasajeros` es la variable que se quiere predecir, utilizar `PAX` como entrada del modelo generaría fuga de información, ya que contiene un valor calculado directamente a partir de la variable objetivo.

Por este motivo, `PAX` no se va a utilizar como variable predictora.

In [41]:
df_vuelos = df_vuelos.drop(columns=["PAX"])

df_vuelos.head()

,Fecha UTC,Hora UTC,Clase de Vuelo (todos los vuelos),Clasificación Vuelo,Tipo de Movimiento,Aeropuerto,Origen / Destino,Aerolinea Nombre,Aeronave,Pasajeros,Calidad dato
6,01/01/2025,0:09,Regular,Doméstico,Despegue,IGU,EZE,AEROLINEAS ARGENTINAS SA,BO-737-8BK,83.0,DEFINITIVO
8,01/01/2025,0:12,Regular,Doméstico,Despegue,DOZ,AER,JETSMART AIRLINES S.A.,AIB-A320-232,23.0,DEFINITIVO
12,01/01/2025,0:20,Regular,Doméstico,Despegue,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,103.0,DEFINITIVO
15,01/01/2025,0:29,Regular,Doméstico,Despegue,IGU,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,104.0,DEFINITIVO
21,01/01/2025,0:44,Regular,Doméstico,Despegue,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,103.0,DEFINITIVO


### 5.7 Revisión de la variable Aeronave

Otra de las variables que se necesita revisar antes de decidir si puede usarse para la predicción es `Aeronave`.

La idea es ver qué información contiene esta columna y cómo están distribuidos sus valores. Después se va a evaluar si el tipo de aeronave puede conocerse antes de la salida del vuelo, que es el momento en el que se plantea hacer la predicción.

In [42]:
df_vuelos["Aeronave"].value_counts().head(20)

Aeronave
EMB-ERJ190100IGW    38518
0                   14335
AIB-A320-232        12370
BO-737-8            11022
BO-737-800           8012
BO-737-8Q8           7634
BO-B737-800          5971
BO-737-8U3           4886
BO-737-8HX           3910
BO-737-8SH           3639
BO-737-86N           3028
BO-B737-8SH          1917
BO-737-8BK           1581
BO-B-737-7BD         1411
BO-B737-86J          1368
BO-B737-8BK          1360
BO-737-7BD           1231
BO-B737-8LP          1224
BO-B737-8MAX         1115
BO-B-737-76N         1097
Name: count, dtype: int64

#### Resultado de la revisión

La variable `Aeronave` contiene códigos que identifican el fabricante y el modelo de avión utilizado en cada operación.

También se observa que una cantidad importante de registros tiene el valor `0`. La documentación oficial consultada no aclara qué representa ese valor, por lo que no se lo puede interpretar directamente como un dato faltante.

Además, como la predicción se plantea antes de la salida del vuelo, todavía es necesario confirmar si la aeronave registrada corresponde a información disponible previamente o al avión que finalmente realizó la operación.

Por este motivo, por ahora `Aeronave` no se incorpora como variable predictora y se mantiene en el dataset hasta definir su tratamiento.

### 5.8 Revisión de Fecha UTC y Hora UTC

Como la predicción se plantea antes de la salida del vuelo, también es necesario revisar qué representan `Fecha UTC` y `Hora UTC`.

Según la documentación oficial de ANAC, estas columnas indican la fecha y la hora del vuelo en horario UTC. Sin embargo, la fuente no aclara si corresponden al horario programado previamente o al momento efectivo en que se realizó la operación.

Esta diferencia es importante porque, si se tratara del horario efectivo, esa información podría no estar disponible al momento de hacer la predicción.

Por este motivo, por ahora `Fecha UTC` y `Hora UTC` se mantienen en el dataset, pero no se consideran confirmadas como variables predictoras hasta poder determinar con mayor precisión cómo se generan esos datos.

### 5.9 Revisión de variables después de los filtros

Después de aplicar los filtros del proyecto, algunas variables pueden haber quedado con un único valor y dejar de aportar información útil para una futura predicción.

Por eso, se revisan nuevamente `Clase de Vuelo (todos los vuelos)`, `Clasificación Vuelo`, `Tipo de Movimiento` y `Calidad dato` antes de decidir si tiene sentido mantenerlas en el dataset preparado.

In [43]:
df_vuelos["Clase de Vuelo (todos los vuelos)"].value_counts()

Clase de Vuelo (todos los vuelos)
Regular    132838
Name: count, dtype: int64

In [44]:
df_vuelos["Clasificación Vuelo"].value_counts()

Clasificación Vuelo
Doméstico    132838
Name: count, dtype: int64

In [45]:
df_vuelos["Tipo de Movimiento"].value_counts()

Tipo de Movimiento
Despegue    132838
Name: count, dtype: int64

In [46]:
df_vuelos["Calidad dato"].value_counts()

Calidad dato
DEFINITIVO    132838
Name: count, dtype: int64

#### Resultado de la revisión

Después de aplicar los filtros, las variables `Clase de Vuelo (todos los vuelos)`, `Clasificación Vuelo` y `Tipo de Movimiento` quedaron con un único valor, ya que todos los registros corresponden a vuelos regulares, domésticos y de despegue.

También se observa que `Calidad dato` tiene el valor `DEFINITIVO` en todos los casos.

Como estas columnas ya no presentan variación dentro del conjunto de datos y no aportarían información para una futura predicción, se eliminan del dataframe que se va a seguir preparando para el modelo.

In [47]:
df_vuelos = df_vuelos.drop(columns=[
    "Clase de Vuelo (todos los vuelos)",
    "Clasificación Vuelo",
    "Tipo de Movimiento",
    "Calidad dato"
])

df_vuelos.head()

,Fecha UTC,Hora UTC,Aeropuerto,Origen / Destino,Aerolinea Nombre,Aeronave,Pasajeros
6,01/01/2025,0:09,IGU,EZE,AEROLINEAS ARGENTINAS SA,BO-737-8BK,83.0
8,01/01/2025,0:12,DOZ,AER,JETSMART AIRLINES S.A.,AIB-A320-232,23.0
12,01/01/2025,0:20,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,103.0
15,01/01/2025,0:29,IGU,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,104.0
21,01/01/2025,0:44,BAR,EZE,FB LÍNEAS AÉREAS - FLYBONDI,BO-737-8Q8,103.0


### 5.10 Estado de las variables después de la preparación

Después de los filtros y transformaciones realizadas, el conjunto de datos quedó formado por 132.838 registros y 7 columnas.

Las variables que todavía forman parte del dataset son:

- `Fecha UTC` y `Hora UTC`: se mantienen, aunque todavía no se consideran confirmadas como predictoras porque la fuente no aclara si corresponden al horario programado o al horario efectivo del vuelo.
- `Aeropuerto`: representa el aeropuerto de salida, ya que el dataset quedó limitado a despegues.
- `Origen / Destino`: representa el aeropuerto de destino.
- `Aerolinea Nombre`: se mantiene como posible variable predictora.
- `Aeronave`: se mantiene en el dataset, pero todavía no se incorpora como predictora porque no se pudo confirmar si esa información está disponible antes de la salida del vuelo.
- `Pasajeros`: es la variable objetivo del proyecto.

## 6. Resultado de la preparación

Después de la carga, revisión y preparación de los datos, el conjunto original de 597.784 registros quedó reducido a 132.838 operaciones correspondientes a vuelos regulares de cabotaje y registradas como despegues.

Durante la preparación se eliminaron registros duplicados exactos y también se descartaron variables que dejaron de aportar información después de los filtros, como `Clase de Vuelo (todos los vuelos)`, `Clasificación Vuelo`, `Tipo de Movimiento` y `Calidad dato`.

También se descartó `PAX` como posible variable predictora, ya que se comprobó que su valor es exactamente la mitad de `Pasajeros`, por lo que utilizarla generaría fuga de información.

El conjunto preparado mantiene las siguientes variables:

- `Aeropuerto`: representa el aeropuerto de salida.
- `Origen / Destino`: representa el aeropuerto de destino.
- `Aerolinea Nombre`: se mantiene como posible variable predictora.
- `Fecha UTC` y `Hora UTC`: se mantienen en el dataset, aunque todavía no se consideran confirmadas como predictoras porque la fuente no aclara si corresponden al horario programado o al horario efectivo del vuelo.
- `Aeronave`: se mantiene, pero tampoco se considera confirmada como predictora hasta poder asegurar que esa información estaría disponible antes de la salida del vuelo.
- `Pasajeros`: es la variable objetivo que se buscará predecir.

De esta manera, una observación del conjunto preparado representa un registro de despegue correspondiente a una operación regular de cabotaje publicada por ANAC, luego de aplicar los filtros y eliminar duplicados exactos.

El dataset queda preparado para continuar con la etapa de selección definitiva de variables y posterior generación del modelo de Aprendizaje Automático.

In [48]:
print("Cantidad final de filas:", df_vuelos.shape[0])
print("Cantidad final de columnas:", df_vuelos.shape[1])

df_vuelos.columns

Cantidad final de filas: 132838
Cantidad final de columnas: 7


Index(['Fecha UTC', 'Hora UTC', 'Aeropuerto', 'Origen / Destino',
       'Aerolinea Nombre', 'Aeronave', 'Pasajeros'],
      dtype='str')

In [49]:
df_vuelos.to_csv(
    "../data/processed/vuelos_regulares_cabotaje_2025_procesado.csv",
    index=False
)